# TRADING INTELLIGENT · CH05 · LABORATOIRE 02
## Du modèle autorégressif aux fenêtres et au Machine Learning financier

**Question directrice.** Comment passer d’une trajectoire chronologique à un problème supervisé dont chaque prévision aurait réellement été possible à sa date ?

**Références :** manuel CH05 §§5.6–5.13 ; continuité CH02 (AR/ARIMA), CH03 (volatilité), CH04 (relations inter-actifs). Données **synthétiques uniquement**, aucun résultat ne valide une stratégie de marché.

**Fonctionnement :** Colab/local sans réseau. Le laboratoire reconstitue lui-même le dataset canonique de 5.10 quand le CSV n’est pas disponible. Les fenêtres de test sont chronologiques et ne servent pas à choisir les modèles. Les exercices facultatifs sont non bloquants et leurs corrigés n’apparaissent que dans MASTER.

**Convention :** décision **après la clôture de t** ; $r_t$ est alors observable ; la cible $r_{t+1}$ ne le sera qu’après la séance suivante. Les petits rendements du tableau manuel sont exprimés en **points de pourcentage**. Le dataset plus long utilise des rendements **décimaux**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier, plot_tree
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.metrics import (mean_absolute_error, mean_squared_error, accuracy_score,
    confusion_matrix, precision_score, recall_score, f1_score, log_loss, brier_score_loss)
from pathlib import Path
SEED=20261116
PETROL='#174B5B'; GOLD='#BA8C4C'; INK='#23343B'; BLUE='#418D9B'
plt.rcParams.update({'figure.figsize':(9,4),'axes.grid':True,'grid.alpha':.16,'axes.spines.top':False,
                     'axes.spines.right':False,'font.size':10})
print('Environnement disponible | numpy',np.__version__,'| pandas',pd.__version__)

## A · Partir d’une série et du temps

**Pourquoi cette étape ?** Le prix et le rendement ne portent pas la même information.

*Référence du manuel : CH01 et 5.6.*

In [ ]:
r_manual=np.array([.4,-.2,.1,.3,-.1,.2,-.4,.5,0,.1],float) # points de % du manuel 5.7
P_manual=100*np.exp(np.r_[0,np.cumsum(r_manual/100)])
fig,axes=plt.subplots(2,1,figsize=(10,6))
axes[0].plot(np.arange(0,11),P_manual,marker='o',color=PETROL);axes[0].set(ylabel='Prix reconstruit',title='Même histoire : niveaux et variations')
axes[1].bar(np.arange(1,11),r_manual,color=[BLUE if r>=0 else GOLD for r in r_manual]);axes[1].set(xlabel='Séance t',ylabel='Rendement log (pt %)');plt.tight_layout();plt.show()
assert np.allclose(100*np.diff(np.log(P_manual)),r_manual)

## B · AR(1) est une régression simple

**Pourquoi cette étape ?** La cible est le rendement futur ; la variable est son prédécesseur disponible.

*Référence du manuel : 5.6.*

In [ ]:
X_ar1=r_manual[:-1].reshape(-1,1);y_ar1=r_manual[1:]
reg_ar1=LinearRegression().fit(X_ar1,y_ar1)
print('AR(1) : prévision(t+1) =',round(reg_ar1.intercept_,4),'+',round(reg_ar1.coef_[0],4),'x rendement(t)')
fig,ax=plt.subplots();ax.scatter(X_ar1[:,0],y_ar1,color=PETROL)
grid=np.linspace(-.5,.55,150);ax.plot(grid,reg_ar1.predict(grid.reshape(-1,1)),color=GOLD)
ax.set(xlabel='r_t (pt %)',ylabel='r_{t+1} (pt %)',title='AR(1) : nuage temporel réinterprété en régression');plt.show()

**À vous — Correspondance AR / ML.** Distinguez dans $r_{t+1}=c+\phi r_t+\epsilon_{t+1}$ : l’entrée, la cible, le coefficient et l’innovation.

In [ ]:
# Votre espace de travail (facultatif pour exécuter la suite)
# Écrivez ici vos essais, puis confrontez-les aux résultats précédents.

## C · AR(3) devient une régression multiple

**Pourquoi cette étape ?** Nous construisons les sept lignes complètes du tableau de la section 5.7.

*Référence du manuel : 5.7.*

In [ ]:
def make_lags(series,L,h=1):
    """Cible = valeur ponctuelle r[t+h], t index Python. Entrées r[t],...,r[t-L+1]."""
    a=np.asarray(series,dtype=float)
    if L<1 or h<1 or len(a)<L+h:raise ValueError('Historique insuffisant ou fenêtre/horizon invalide')
    t_index=np.arange(L-1,len(a)-h)
    X=np.array([[a[t-j] for j in range(L)] for t in t_index])
    y=a[t_index+h]
    return X,y,t_index+1 # indices de séance mathématiques, 1..T
X3,y3,t3=make_lags(r_manual,3,1)
table=pd.DataFrame(X3,columns=['r_t','r_t-1','r_t-2'],index=pd.Index(t3,name='décision t'))
table['cible_r_t+1']=y3
display(table)
assert X3.shape==(7,3) and y3.shape==(7,)
assert np.allclose(X3[0],[.1,-.2,.4]) and y3[0]==.3
assert np.allclose(X3[-1],[0,.5,-.4]) and np.isclose(y3[-1],.1)

## D · Faire glisser la fenêtre : regarder ce qui entre et sort

**Pourquoi cette étape ?** Chaque date apporte une nouvelle valeur, pas un nouvel historique indépendant.

*Référence du manuel : 5.7.*

In [ ]:
fig,axes=plt.subplots(3,1,figsize=(10,5.7))
for ax,t in zip(axes,[3,4,5]):
    ax.bar(np.arange(1,11),r_manual,color=['#DDE5E8' if k<t-2 or k>t else BLUE for k in range(1,11)])
    ax.axvline(t+.5,color=PETROL,ls='--');ax.axvline(t+1,color=GOLD,ls=':',alpha=.65)
    ax.text(7,.39,f'X_{t}=(r{t},r{t-1},r{t-2}) ; y=r{t+1}',fontsize=10)
    ax.set(xlim=(.3,10.7),ylim=(-.55,.6),ylabel='pt %');ax.set_yticks([-.4,0,.4])
axes[-1].set_xlabel('Séances (la cible t+1 est à droite de la décision)')
plt.tight_layout();plt.show()

**Observation cruciale.** La dernière fenêtre $X_{10}=(r_{10},r_9,r_8)$ est utilisable pour prédire $r_{11}$, mais $r_{11}$ **n’existe pas** dans notre tableau de dix observations. Elle ne constitue donc pas une huitième ligne étiquetée.

In [ ]:
last_window=r_manual[-1:-4:-1]
assert np.allclose(last_window,[.1,0,.5]);print('Fenêtre de prévision non étiquetée X₁₀ =',last_window)

**À vous — Fenêtres.** Écrivez $X_6,y_6,X_7,y_7$ du tableau manuel puis indiquez les observations communes.

In [ ]:
# Votre espace de travail (facultatif pour exécuter la suite)
# Écrivez ici vos essais, puis confrontez-les aux résultats précédents.

## E · AR(p) et taille de l’historique

**Pourquoi cette étape ?** Augmenter le nombre de retards diminue le nombre d’exemples étiquetés disponibles.

*Référence du manuel : 5.6–5.7.*

In [ ]:
for L in [1,3,4,5]:
    XX,yy,tt=make_lags(r_manual,L)
    print(f'L={L} → X{XX.shape}, y{yy.shape}, première décision t={tt[0]}')
assert make_lags(r_manual,5)[0].shape==(5,5)
reg_ar3=LinearRegression().fit(X3[:4],y3[:4]) # démonstration, petit échantillon uniquement
print('AR(3) ajusté sur 4 lignes :',np.round(reg_ar3.coef_,3),'| avertissement: échantillon minuscule')

## F · Lookback L et horizon h : deux axes différents

**Pourquoi cette étape ?** Une cible ponctuelle et une cible cumulée ne décrivent pas le même événement.

*Référence du manuel : 5.8.*

In [ ]:
Xh,yh,th=make_lags(r_manual,L=3,h=3)
print('Cible ponctuelle à t+3 : X',Xh.shape,'| y',yh.shape,'| premières dates',th[:3])
assert len(yh)==len(r_manual)-3-3+1
prices=100*np.exp(np.r_[0,np.cumsum(r_manual/100)])
# prix indexé P_t à t=0..10 ; log P_{t+h}/P_t exprimé en points de %.
ycum=np.array([100*np.log(prices[t+3]/prices[t]) for t in th])
compar=pd.DataFrame({'décision t':th,'r_{t+3} (pt %)':yh,'cumulé t→t+3 (pt %)':ycum})
display(compar.round(4))
assert not np.allclose(yh,ycum)

In [ ]:
fig,ax=plt.subplots(figsize=(10,3.8));xidx=np.arange(len(th));w=.35
ax.bar(xidx-w/2,yh,width=w,color=PETROL,label='Ponctuel r[t+3]')
ax.bar(xidx+w/2,ycum,width=w,color=GOLD,label='Cumulé t→t+3')
ax.axhline(0,color=INK,lw=.8);ax.set_xticks(xidx,th);ax.set(xlabel='Date de décision t',ylabel='Rendement (pt %)',title='Même horizon, questions différentes');ax.legend();plt.show()

**À vous — Indexation.** Sur dix observations brutes, pourquoi $L=3,h=3$ produit-il cinq lignes, et pas sept ?

In [ ]:
# Votre espace de travail (facultatif pour exécuter la suite)
# Écrivez ici vos essais, puis confrontez-les aux résultats précédents.

## G · Du tableau 2D au tenseur séquentiel 3D

**Pourquoi cette étape ?** Les modèles tabulaires utilisent une ligne aplatie ; les modèles séquentiels peuvent conserver temps × variables.

*Référence du manuel : 5.8, préparation de CH07.*

In [ ]:
rng=np.random.default_rng(SEED)
raw=np.column_stack([rng.normal(0,.01,25),rng.normal(0,.015,25)]) # 25 dates, 2 features
L,F=5,raw.shape[1]
sequence=np.array([raw[t-L+1:t+1] for t in range(L-1,len(raw)-1)])
flat=sequence.reshape(len(sequence),L*F)
assert sequence.shape==(20,5,2) and flat.shape==(20,10)
print('tenseur N,L,F:',sequence.shape,'| tableau N,L×F:',flat.shape)

## H · Dataset financier canonique : les six features de 5.10

**Pourquoi cette étape ?** Nous réutilisons strictement les noms, fenêtres et générateur déterministe du manuel.

*Référence du manuel : 5.9–5.10.*

In [ ]:
# Source : scripts/experiment_ch05_section_5_10.py du manuel v2.
# Le fichier local est facultatif, aucun service internet sollicité.
DATASET_ID='TI_CH05_SEC510_Synthese_v2'
FNAME='TI_CH05_SEC510_Synthese_v2.csv'
def generer_jeu_canonique():
    SEED=20261116
    rng=np.random.default_rng(SEED)
    n=570
    rb=np.zeros(n); ra=np.zeros(n)
    volumes=np.exp(rng.normal(10.7, 0.23,n))
    noise=rng.normal(0,.0044,n)
    for t in range(1,n):
        rb[t]=.28*rb[t-1]+rng.normal(0,.007)
        if t>=4:
            vrel=volumes[t-1]/np.mean(volumes[t-4:t-1])
            mom=ra[t-1]+ra[t-2]+ra[t-3]
            ra[t]=(.32*ra[t-1]+.24*rb[t-1]+.0008*np.tanh(mom/.012)
                +.00065*(vrel-1)+noise[t])
        else:
            ra[t]=noise[t]
    P=100*np.exp(np.cumsum(ra)); B=80*np.exp(np.cumsum(rb))
    idx=pd.bdate_range('2024-01-02',periods=n)
    S=pd.DataFrame({'P':P,'B':B,'V':volumes,'rA':ra,'rB':rb},index=idx)
    X=pd.DataFrame(index=idx)
    X['retard']=S.rA
    X['momentum3']=S.rA.rolling(3).sum()
    X['volatilite5']=S.rA.rolling(5).std(ddof=1)
    X['volume_relatif']=S.V/S.V.shift(1).rolling(5).mean()
    X['ecart_AB']=S.rA-S.rB
    X['ecart_sma3']=S.P/S.P.rolling(3).mean()-1
    X['cible_rendement_t1']=S.rA.shift(-1)
    jeu=X.dropna().copy()
    return jeu.rename_axis("decision_date")

file_candidates=[Path(FNAME),Path('datasets')/FNAME,Path('..')/'datasets'/FNAME]
found=next((p for p in file_candidates if p.is_file()),None)
if found is None:
    jeu=generer_jeu_canonique(); print('CSV absent : dataset canonique régénéré avec SEED=20261116')
else:
    jeu=pd.read_csv(found,parse_dates=['decision_date']).set_index('decision_date')
    print('Dataset canonique lu localement :',found)
FEATURES=['retard','momentum3','volatilite5','volume_relatif','ecart_AB','ecart_sma3']
TARGET='cible_rendement_t1'
assert len(jeu)==564 and jeu[FEATURES+[TARGET]].notna().all().all()
assert np.isclose(jeu.iloc[0]['retard'], -.005122777294357389)
print('Dimensions:',jeu.shape,'| premières lignes :')
display(jeu.head(3))

## I · Tester la causalité des features

**Pourquoi cette étape ?** Une caractéristique doit être calculable avec l’information effectivement disponible après t.

*Référence du manuel : 5.9, 5.13.*

In [ ]:
# La ligne t contient rA_t et la cible rA_{t+1}. La valeur rA_{t+1} ne figure dans aucune feature.
assert np.allclose(jeu[TARGET].to_numpy()[:-1],jeu['retard'].to_numpy()[1:])
assert jeu.index.is_monotonic_increasing and jeu.index.is_unique
fig,axes=plt.subplots(2,1,figsize=(10,6),sharex=True)
axes[0].plot(jeu.index,100*jeu['retard'],color=PETROL,lw=.8,label='rendement observé r_t')
axes[0].plot(jeu.index,100*jeu[TARGET],color=GOLD,lw=.75,alpha=.6,label='cible r_{t+1}')
axes[0].legend();axes[0].set_ylabel('Points de %')
axes[1].plot(jeu.index,100*jeu['volatilite5'],color=BLUE);axes[1].set(xlabel='Dates de décision',ylabel='Volatilité 5 (pt %)',title='Feature calculée sur une fenêtre passée');plt.tight_layout();plt.show()

**À vous — Audit de variable.** Pourquoi une moyenne mobile centrée, ou une publication macroéconomique datée t mais connue à t+2, ne sont-elles pas des features admissibles à t ?

In [ ]:
# Votre espace de travail (facultatif pour exécuter la suite)
# Écrivez ici vos essais, puis confrontez-les aux résultats précédents.

## J · Train / validation / test : ne pas changer les dates

**Pourquoi cette étape ?** La séparation chronologique est fixée avant d’examiner les scores des modèles.

*Référence du manuel : 5.10, 5.13.*

In [ ]:
N=len(jeu);ntr=284;nv=140
splits={'train':slice(0,ntr),'validation':slice(ntr,ntr+nv),'test':slice(ntr+nv,N)}
Xfull=jeu[FEATURES].to_numpy();yfull=jeu[TARGET].to_numpy()
Xsets={k:Xfull[s] for k,s in splits.items()};ysets={k:yfull[s] for k,s in splits.items()}
for k,s in splits.items():print(k,len(jeu.iloc[s]),'décisions :',jeu.index[s][0].date(),'→',jeu.index[s][-1].date())
assert [len(ysets[k]) for k in ['train','validation','test']]==[284,140,140]
assert jeu.index[ntr-1]<jeu.index[ntr]<jeu.index[ntr+nv]
# Convention explicite du protocole : chaque décision est prise APRÈS la clôture.
# La cible de la dernière ligne de TRAIN est r(t+1) au jour de la première
# décision VALIDATION. On ajuste donc le modèle après cette clôture,
# immédiatement avant de prédire le lendemain de cette même séance.
# Si l'on exige un modèle déjà figé avant cette clôture, on exclut la
# dernière ligne : train_effectif = slice(0,ntr-1).
last_train_label_available_at=jeu.index[ntr]
first_validation_decision_at=jeu.index[ntr]
assert last_train_label_available_at==first_validation_decision_at
print('Horloge du protocole : ajustement après la clôture du',first_validation_decision_at.date(),
      'avant la prévision de t+1 ; dernière cible TRAIN mature à cet instant.')
print('Option modèle figé AVANT cette clôture : retirer la dernière ligne TRAIN.')
fig,ax=plt.subplots(figsize=(10,2))
for left,right,label,color in [(0,284,'TRAIN',PETROL),(284,424,'VALIDATION',GOLD),(424,564,'TEST',BLUE)]:
    ax.barh(0,width=right-left,left=left,height=.65,color=color,label=label)
ax.set(xlim=(0,N),ylim=(-.6,.6),xlabel='Lignes de décisions ordonnées',yticks=[],title='Une validation chronologique, jamais aléatoire');ax.legend(ncol=3,loc='upper center');plt.show()

## K · Régression : baseline, OLS, Ridge, Lasso et arbre

**Pourquoi cette étape ?** Même cible ; même échantillon ; même protocole. La pénalité est choisie exclusivement sur validation.

*Référence du manuel : 5.10–5.12.*

In [ ]:
regressors={'Zéro':DummyRegressor(strategy='constant',constant=0.),
 'OLS':make_pipeline(StandardScaler(),LinearRegression()),
 'Ridge(1)':make_pipeline(StandardScaler(),Ridge(alpha=1.)),
 'Ridge(40)':make_pipeline(StandardScaler(),Ridge(alpha=40.)),
 'Lasso(.00010)':make_pipeline(StandardScaler(),Lasso(alpha=.00010,max_iter=40000)),
 'Lasso(.00035)':make_pipeline(StandardScaler(),Lasso(alpha=.00035,max_iter=40000)),
 'Arbre(2)':DecisionTreeRegressor(max_depth=2,min_samples_leaf=24,random_state=SEED),
 'Arbre(7)':DecisionTreeRegressor(max_depth=7,min_samples_leaf=4,random_state=SEED)}
reg_rows=[]
for name,model in regressors.items():
    model.fit(Xsets['train'],ysets['train'])
    for part in ['train','validation']:
        yh=model.predict(Xsets[part]);reg_rows.append({'modèle':name,'période':part,
              'MAE (pt %)':100*mean_absolute_error(ysets[part],yh),
              'RMSE (pt %)':100*np.sqrt(mean_squared_error(ysets[part],yh))})
score_reg=pd.DataFrame(reg_rows)
display(score_reg.pivot(index='modèle',columns='période',values='MAE (pt %)').round(4))
assert np.isclose(score_reg.query('modèle=="OLS" and période=="validation"')['MAE (pt %)'].iloc[0],.3195921155531352,atol=1e-5)

In [ ]:
fig,ax=plt.subplots(figsize=(10,3.5));sub=score_reg.query('période=="validation"').sort_values('MAE (pt %)')
ax.bar(sub['modèle'],sub['MAE (pt %)'],color=PETROL);ax.set(ylabel='MAE validation (pt %)',title='Sélectionner sur validation, pas sur test');ax.tick_params(axis='x',rotation=35);plt.tight_layout();plt.show()

**À vous — Sélection.** Choisissez Ridge et Lasso en ne consultant que leur MAE de validation. Pourquoi n’est-il pas licite de choisir ensuite celui dont la MAE de test est la plus faible ?

In [ ]:
# Votre espace de travail (facultatif pour exécuter la suite)
# Écrivez ici vos essais, puis confrontez-les aux résultats précédents.

## L · Classification : une cible différente, mêmes features

**Pourquoi cette étape ?** L’étiquette vaut 1 pour une hausse stricte du rendement suivant, 0 sinon.

*Référence du manuel : 5.4, 5.10, 5.12.*

In [ ]:
binary={k:(v>0).astype(int) for k,v in ysets.items()}
cls_models={'Fréquence train':DummyClassifier(strategy='prior'),
            'Logistique':make_pipeline(StandardScaler(),LogisticRegression(C=1.,max_iter=2000)),
            'Arbre classif(2)':DecisionTreeClassifier(max_depth=2,min_samples_leaf=24,random_state=SEED)}
class_rows=[]
for name,model in cls_models.items():
    model.fit(Xsets['train'],binary['train'])
    for part in ['train','validation']:
        p=model.predict_proba(Xsets[part])[:,1]
        class_rows.append({'modèle':name,'période':part,'accuracy':accuracy_score(binary[part],p>=.5),
         'Brier':brier_score_loss(binary[part],p),'log-loss':log_loss(binary[part],p,labels=[0,1])})
display(pd.DataFrame(class_rows).query('période=="validation"').round(4))
assert np.allclose(cls_models['Logistique'].predict_proba(Xsets['validation']).sum(axis=1),1)

## M · Vérifier les probabilités, pas seulement les classes

**Pourquoi cette étape ?** Une classification peut devenir illusoire lorsque le taux de réussite majoritaire domine.

*Référence du manuel : 5.12.*

In [ ]:
p=cls_models['Logistique'].predict_proba(Xsets['validation'])[:,1]
fig,axes=plt.subplots(1,2,figsize=(11,3.6))
axes[0].hist(p,bins=14,color=PETROL,alpha=.8);axes[0].set(xlabel='p(hausse)',ylabel='Comptage',title='Distribution des probabilités')
for th in [.3,.5,.7]:
    ypred=p>=th
    axes[1].scatter([th],[recall_score(binary['validation'],ypred,zero_division=0)],s=100,label=f'seuil {th}')
axes[1].set(xlim=(.2,.8),ylim=(-.05,1.05),xlabel='Seuil',ylabel='Rappel',title='Changer le seuil ≠ réapprendre');axes[1].legend();plt.tight_layout();plt.show()

## N · Contrôle : processus prévisible versus bruit

**Pourquoi cette étape ?** Une méthode doit réussir sur un signal contrôlé et ne pas inventer un signal absent.

*Référence du manuel : 5.11–5.12.*

In [ ]:
rng=np.random.default_rng(SEED+40);n=1250
signal=np.zeros(n);noise=rng.normal(0,.01,n)
for t in range(1,n):signal[t]=.7*signal[t-1]+noise[t]
white=rng.normal(0,.01,n)
exp_rows=[]
for name,series in [('AR(1) signal',signal),('Bruit indépendant',white)]:
    XX,yy,idx=make_lags(series,1)
    cut=int(.7*len(yy));model=LinearRegression().fit(XX[:cut],yy[:cut]);pr=model.predict(XX[cut:])
    mse=mean_squared_error(yy[cut:],pr);mse0=mean_squared_error(yy[cut:],np.zeros(len(pr)))
    exp_rows.append([name,model.coef_[0],mse,mse0,1-mse/mse0])
display(pd.DataFrame(exp_rows,columns=['processus','pente apprise','MSE test','MSE zéro','gain sur baseline']).round(5))
assert exp_rows[0][-1]>0.20
assert abs(exp_rows[1][-1])<.1

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,3.3))
for ax,serie,label in zip(axes,[signal,white],['Dépendance réelle','Bruit sans dépendance']) :
    ax.scatter(serie[:-1],serie[1:],alpha=.15,s=10,color=PETROL);ax.set(xlabel='r_t',ylabel='r_{t+1}',title=label)
plt.tight_layout();plt.show()

## O · Voir la fuite : le futur introduit dans X

**Pourquoi cette étape ?** Une performance parfaite peut être frauduleuse au sens scientifique, même sans bug Python.

*Référence du manuel : 5.13.*

In [ ]:
rng=np.random.default_rng(SEED+5);rr=rng.normal(0,.01,2000);y_bad=(rr[1:]>0).astype(int)
X_past=rr[:-1].reshape(-1,1) # autorisé à t
X_future=rr[1:].reshape(-1,1) # interdit : c'est la cible sous sa forme numérique
cut=1400
m_ok=LogisticRegression(max_iter=1000).fit(X_past[:cut]*1000,y_bad[:cut])
m_leak=LogisticRegression(max_iter=1000).fit(X_future[:cut]*1000,y_bad[:cut])
a_ok=accuracy_score(y_bad[cut:],m_ok.predict(X_past[cut:]*1000))
a_leak=accuracy_score(y_bad[cut:],m_leak.predict(X_future[cut:]*1000))
print('Accuracy prédiction causale :',round(a_ok,3),'| prédiction avec futur :',round(a_leak,3))
assert a_leak>.99 and .43<a_ok<.57

Cette expérience ne justifie pas l’utilisation d’une telle variable. Elle enseigne à détecter une erreur de construction : `shift(-1)` peut être correct pour créer **y**, jamais pour une **feature disponible à t**. Les colonnes peuvent avoir des noms innocents et contenir malgré tout le futur.

## P · Standardisation : apprendre seulement sur TRAIN

**Pourquoi cette étape ?** Une moyenne ou variance globale contient des renseignements sur les observations futures.

*Référence du manuel : 5.13.*

In [ ]:
std_train=StandardScaler().fit(Xsets['train'])
std_global=StandardScaler().fit(Xfull)
print('Moyenne feature 1 sur TRAIN:',round(std_train.mean_[0],6),'| globale:',round(std_global.mean_[0],6))
Ztrain=std_train.transform(Xsets['train']);Ztest=std_train.transform(Xsets['test'])
assert np.allclose(Ztrain.mean(axis=0),0,atol=1e-10)
assert np.allclose(Ztest,std_train.transform(Xsets['test']))
# La vraie procédure utilise une Pipeline pour empêcher l'oubli.
print('La transformation du TEST réutilise exclusivement les paramètres appris sur TRAIN.')

## Q · Maturité des labels et chevauchement

**Pourquoi cette étape ?** À une coupure de formation, certaines cibles futures ne sont pas encore observables.

*Référence du manuel : 5.8 et 5.13.*

In [ ]:
T=40;L=5;h=4;rng=np.random.default_rng(777)
series=rng.normal(size=T);XX,yy,decision=make_lags(series,L,h)
# 'decision' numérote les séances 1..T ; la cible ne devient disponible qu'en (t+h).
available_at=decision+h
first_validation_decision=25
eligible=decision[(decision<first_validation_decision)&(available_at<=first_validation_decision)]
not_yet=decision[(decision<first_validation_decision)&(available_at>first_validation_decision)]
print('Dates formation autorisées à décision t=25 :',eligible[-5:])
print('Dates de formation dont la cible n’est PAS mature :',not_yet)
assert all((not_yet+h)>first_validation_decision)
fig,ax=plt.subplots(figsize=(10,2.8))
for d in [21,22,23,24]:
    ax.plot([d,d+h],[d,d],marker='o',color=GOLD if d+h>25 else PETROL,lw=3)
ax.axvline(25,color=INK,ls='--');ax.set(xlabel='Séances',ylabel='Date de décision t',title='Horizon h=4 : la cible chevauche la coupure');plt.show()

## R · Pipeline intégrateur et audit final

**Pourquoi cette étape ?** Nous séparons ce qui est appris, ce qui est prévu et ce qui est vérifié.

*Référence du manuel : 5.5–5.13.*

In [ ]:
# Sélection du modèle sur VALIDATION uniquement (test non consulté plus haut).
best_name=min(['OLS','Ridge(1)','Ridge(40)','Lasso(.00010)','Lasso(.00035)','Arbre(2)'],
              key=lambda name:float(score_reg.query('modèle==@name and période=="validation"')['MAE (pt %)'].iloc[0]))
best_model=regressors[best_name]
# Évaluation finale de la CLASSIFICATION : comparaison des modèles et des
# probabilités choisies en validation, puis ouverture unique du test.
class_valid=pd.DataFrame(class_rows).query('période=="validation"')
selected_class_name=min(class_valid['modèle'],
        key=lambda name: float(class_valid.loc[class_valid['modèle']==name,'Brier'].iloc[0]))
selected_class_model=cls_models[selected_class_name]
prob_test=selected_class_model.predict_proba(Xsets['test'])[:,1]
from sklearn.metrics import roc_auc_score
acc_final=accuracy_score(binary['test'],prob_test>=.5)
brier_final=brier_score_loss(binary['test'],prob_test)
ll_final=log_loss(binary['test'],prob_test,labels=[0,1])
auc_final=roc_auc_score(binary['test'],prob_test) if len(np.unique(binary['test']))==2 else float('nan')
print('CLASSIFICATION modèle sélectionné sur validation (Brier) :',selected_class_name)
print('TEST classification : accuracy=',round(acc_final,4),'Brier=',round(brier_final,4),
      'log-loss=',round(ll_final,4),'ROC-AUC=',round(auc_final,4))
assert np.all((0<=prob_test)&(prob_test<=1))
final_mae=mean_absolute_error(ysets['test'],best_model.predict(Xsets['test']))
base_mae=mean_absolute_error(ysets['test'],np.zeros(len(ysets['test'])))
print('Modèle sélectionné sur validation :',best_name)
print('Test final : MAE modèle',round(100*final_mae,4),'pt %, baseline zéro',round(100*base_mae,4),'pt %')
print('Le test est ouvert UNE seule fois après le choix du modèle.')
assert len(Xsets['test'])==140 and np.isfinite(final_mae)
print('CONTRÔLES NB02 : OK — jeu canonique, cibles, horizons, maturité, splits, classification finale, pas de fit sur TEST.')

**À vous — Projet de transfert.** Imaginez une variable macroéconomique datée t et publiée à t+2. Comment mettrez-vous à jour son horodatage et l’alignement des features sans regarder le futur ?

In [ ]:
# Votre espace de travail (facultatif pour exécuter la suite)
# Écrivez ici vos essais, puis confrontez-les aux résultats précédents.

### Mini-projet à rendre — NB02 (sans corrigé dans STUDENT)

**Livrable A (figure + tableau)** : reconstruire de mémoire les trois dernières fenêtres du jeu de dix rendements et justifier chaque cible avec ses indices.

**Livrable B (code + test)** : créer la même matrice pour `L=5`, `h=3`, puis tester par assertions que `X[-1,0]` et `y[-1]` pointent vers les dates annoncées.

**Livrable C (note méthodologique, 150 mots)** : expliquer à quelle clôture les étiquettes d'entraînement sont connues et comment vous excluriez une ligne si le modèle devait être figé plus tôt.

**Livrable D (tableau d'évaluation)** : consigner MAE du modèle et de la baseline ainsi que Brier, accuracy et log-loss du classifieur final ; indiquer explicitement quel critère a servi à la sélection et sur quelle période.

**Critère d'acceptation :** aucune référence à une observation future dans une feature, résultats reproductibles et explication claire des unités.

### Checklist finale — NB02
- [ ] Je sais construire AR(1), AR(p) et la matrice de retards manuellement.
- [ ] Je distingue lookback, rolling, horizon, fenêtre de formation et fenêtre de test.
- [ ] Je sais calculer $N=T-L-h+1$ pour une cible ponctuelle $t+h$.
- [ ] Je distingue une cible ponctuelle d’un rendement cumulé.
- [ ] Je peux produire une matrice 2D et un tenseur 3D sans confusion de dimensions.
- [ ] Chaque feature est disponible à la décision, et chaque label est mature au moment de l’entraînement.
- [ ] Le scaling et la sélection ne voient jamais le test.
- [ ] Je sais expliquer pourquoi un score artificiellement parfait peut révéler une fuite.
- [ ] Je sépare qualité de prévision, probabilité et profit net.

**Vers CH06 :** validation temporelle approfondie (walk-forward, purge et embargo si nécessaire, recherche d’hyperparamètres sans contamination). **Vers CH07 :** séquences 3D et réseaux neuronaux. **Vers CH08–CH09 :** décision, coûts, portefeuille et backtesting.